In [1]:
"""
Complete GRPO Training Loop for Requirements Engineering
Based on DeepSeek-R1 methodology adapted for ISO 29148 compliance
"""

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import List, Dict, Tuple, Optional
import numpy as np
from dataclasses import dataclass
from tqdm import tqdm
import wandb
from pathlib import Path
import json

# Import the ISO compliance reward function
from iso_compliance_reward import compute_iso_compliance_reward


@dataclass
class GRPOConfig:
    """GRPO hyperparameters based on DeepSeek-R1"""
    # Model configuration
    model_name: str = "Qwen/Qwen2.5-7B-Instruct"
    
    # GRPO hyperparameters (from DeepSeek-R1)
    learning_rate: float = 3e-6
    kl_coef: float = 0.001
    clip_ratio: float = 10.0
    temperature: float = 1.0
    
    # Group sampling
    group_size: int = 16
    num_questions_per_step: int = 32
    batch_size: int = 512
    
    # Training configuration
    max_steps: int = 2000
    ref_model_update_interval: int = 400
    max_seq_length: int = 2048
    
    # Reward configuration
    reward_threshold: float = 0.8
    
    # Optimization
    inner_epochs: int = 1
    num_minibatches: int = 16
    gradient_accumulation_steps: int = 1
    max_grad_norm: float = 1.0
    
    # Evaluation
    eval_interval: int = 100
    eval_samples: int = 100
    save_interval: int = 400
    
    # Logging
    wandb_project: str = "grpo-requirements"
    output_dir: str = "./grpo_outputs"


class RequirementsDataset(Dataset):
    """Dataset for vague requirements"""
    
    def __init__(self, data_path: str, tokenizer, max_length: int = 512):
        """
        data_path: JSON file with format:
        [
            {
                "vague_requirement": "The system should be fast.",
                "iso_compliant_requirement": "The system shall process...",
                "requirement_id": "REQ-001"
            },
            ...
        ]
        """
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        with open(data_path, 'r') as f:
            self.data = json.load(f)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        vague_req = item['vague_requirement']
        
        # Create prompt template
        prompt = self._create_prompt(vague_req)
        
        return {
            'vague_requirement': vague_req,
            'prompt': prompt,
            'requirement_id': item.get('requirement_id', f'REQ-{idx:04d}'),
            'reference_iso': item.get('iso_compliant_requirement', '')
        }
    
    def _create_prompt(self, vague_req: str) -> str:
        """Create structured prompt for requirements transformation"""
        template = f"""Transform the following vague requirement into an ISO 29148-compliant requirement that is accurate, concise, unambiguous, complete, and verifiable.

Vague Requirement: {vague_req}

ISO-Compliant Requirement:"""
        return template


class GRPOTrainer:
    """GRPO Trainer for Requirements Engineering"""
    
    def __init__(self, config: GRPOConfig):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # Initialize models
        print(f"Loading model: {config.model_name}")
        self.policy_model = AutoModelForCausalLM.from_pretrained(
            config.model_name,
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )
        
        # Reference model (frozen copy)
        self.ref_model = AutoModelForCausalLM.from_pretrained(
            config.model_name,
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )
        self.ref_model.eval()
        for param in self.ref_model.parameters():
            param.requires_grad = False
        
        # Tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(config.model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        # Optimizer
        self.optimizer = torch.optim.AdamW(
            self.policy_model.parameters(),
            lr=config.learning_rate,
            betas=(0.9, 0.95)
        )
        
        # Training state
        self.global_step = 0
        self.best_eval_score = 0.0
        
        # Create output directory
        Path(config.output_dir).mkdir(parents=True, exist_ok=True)
        
        # Initialize wandb
        wandb.init(
            project=config.wandb_project,
            config=vars(config),
            name=f"grpo-requirements-{config.model_name.split('/')[-1]}"
        )
    
    def compute_log_probs(
        self,
        model: AutoModelForCausalLM,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor
    ) -> torch.Tensor:
        """Compute log probabilities for generated sequences"""
        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_dict=True
            )
            logits = outputs.logits
            
            # Shift for causal LM: predict next token
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = input_ids[:, 1:].contiguous()
            
            # Compute log probabilities
            log_probs = F.log_softmax(shift_logits, dim=-1)
            
            # Gather log probs for actual tokens
            token_log_probs = torch.gather(
                log_probs,
                dim=-1,
                index=shift_labels.unsqueeze(-1)
            ).squeeze(-1)
            
            # Mask padding tokens
            mask = attention_mask[:, 1:].contiguous()
            token_log_probs = token_log_probs * mask
            
            # Sum over sequence
            sequence_log_probs = token_log_probs.sum(dim=-1)
            
        return sequence_log_probs
    
    def sample_group(
        self,
        prompts: List[str],
        num_samples: int
    ) -> Tuple[List[List[str]], List[torch.Tensor], List[torch.Tensor]]:
        """
        Sample G outputs for each prompt
        Returns: (generated_texts, input_ids, attention_masks)
        """
        all_generated = []
        all_input_ids = []
        all_attention_masks = []
        
        for prompt in prompts:
            generated_texts = []
            input_ids_list = []
            attention_masks_list = []
            
            # Tokenize prompt
            prompt_inputs = self.tokenizer(
                prompt,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=self.config.max_seq_length // 2
            ).to(self.device)
            
            # Generate G samples
            for _ in range(num_samples):
                with torch.no_grad():
                    outputs = self.policy_model.generate(
                        **prompt_inputs,
                        max_new_tokens=self.config.max_seq_length // 2,
                        temperature=self.config.temperature,
                        do_sample=True,
                        top_p=0.95,
                        pad_token_id=self.tokenizer.pad_token_id,
                        eos_token_id=self.tokenizer.eos_token_id
                    )
                
                # Decode
                full_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
                
                # Extract generated requirement (after prompt)
                generated_text = full_text[len(prompt):].strip()
                generated_texts.append(generated_text)
                
                # Store full sequence for log prob computation
                input_ids_list.append(outputs[0])
                
                # Create attention mask
                attention_mask = torch.ones_like(outputs[0])
                attention_masks_list.append(attention_mask)
            
            all_generated.append(generated_texts)
            all_input_ids.append(input_ids_list)
            all_attention_masks.append(attention_masks_list)
        
        return all_generated, all_input_ids, all_attention_masks
    
    def compute_rewards(
        self,
        generated_groups: List[List[str]]
    ) -> Tuple[List[List[float]], Dict]:
        """
        Compute ISO compliance rewards for all generated outputs
        Returns: (rewards, statistics)
        """
        all_rewards = []
        stats = {
            'mean_reward': 0.0,
            'std_reward': 0.0,
            'pass_rate': 0.0,
            'mean_rules_passed': 0.0
        }
        
        total_samples = 0
        total_reward = 0.0
        total_passed = 0
        total_rules_passed = 0
        
        for group in generated_groups:
            group_rewards = []
            
            for generated_req in group:
                # Compute ISO compliance reward
                reward, rule_results = compute_iso_compliance_reward(generated_req)
                group_rewards.append(reward)
                
                # Update statistics
                total_reward += reward
                total_samples += 1
                if reward >= self.config.reward_threshold:
                    total_passed += 1
                total_rules_passed += sum(rule_results.values())
            
            all_rewards.append(group_rewards)
        
        # Compute statistics
        stats['mean_reward'] = total_reward / total_samples if total_samples > 0 else 0.0
        stats['pass_rate'] = total_passed / total_samples if total_samples > 0 else 0.0
        stats['mean_rules_passed'] = total_rules_passed / total_samples if total_samples > 0 else 0.0
        
        # Compute std across all rewards
        flat_rewards = [r for group in all_rewards for r in group]
        stats['std_reward'] = np.std(flat_rewards) if len(flat_rewards) > 1 else 0.0
        
        return all_rewards, stats
    
    def compute_advantages(
        self,
        rewards: List[List[float]]
    ) -> List[List[float]]:
        """
        Compute advantages using group normalization (Equation 3 from paper)
        A_i = (r_i - mean(r_1,...,r_G)) / std(r_1,...,r_G)
        """
        advantages = []
        
        for group_rewards in rewards:
            mean_reward = np.mean(group_rewards)
            std_reward = np.std(group_rewards)
            
            # Prevent division by zero
            if std_reward < 1e-8:
                std_reward = 1.0
            
            group_advantages = [
                (r - mean_reward) / std_reward
                for r in group_rewards
            ]
            advantages.append(group_advantages)
        
        return advantages
    
    def compute_kl_divergence(
        self,
        policy_log_probs: torch.Tensor,
        ref_log_probs: torch.Tensor
    ) -> torch.Tensor:
        """
        Compute KL divergence: D_KL(π_θ || π_ref)
        From Equation 2: π_θ(o|q)/π_ref(o|q) - log(π_θ(o|q)/π_ref(o|q)) - 1
        """
        # Log ratio
        log_ratio = policy_log_probs - ref_log_probs
        
        kl = torch.exp(log_ratio) - log_ratio - 1
        
        return kl
    
    def compute_grpo_loss(
        self,
        policy_log_probs: torch.Tensor,
        ref_log_probs: torch.Tensor,
        old_log_probs: torch.Tensor,
        advantages: torch.Tensor
    ) -> Tuple[torch.Tensor, Dict]:
        """
        Compute GRPO loss (Equation 1 from paper)
        
        L_GRPO = -E[min(ratio * A, clip(ratio, 1-ε, 1+ε) * A) - β * D_KL]
        where ratio = π_θ(o|q) / π_θ_old(o|q)
        """
        # Importance ratio
        log_ratio = policy_log_probs - old_log_probs
        ratio = torch.exp(log_ratio)
        
        # Clipped ratio
        clip_ratio = torch.clamp(
            ratio,
            1.0 - self.config.clip_ratio,
            1.0 + self.config.clip_ratio
        )
        
        # Policy loss: -min(ratio * A, clip(ratio) * A)
        policy_loss_1 = ratio * advantages
        policy_loss_2 = clip_ratio * advantages
        policy_loss = -torch.min(policy_loss_1, policy_loss_2).mean()
        
        # KL divergence penalty
        kl = self.compute_kl_divergence(policy_log_probs, ref_log_probs)
        kl_penalty = self.config.kl_coef * kl.mean()
        
        # Total loss
        total_loss = policy_loss + kl_penalty
        
        # Statistics
        stats = {
            'loss/total': total_loss.item(),
            'loss/policy': policy_loss.item(),
            'loss/kl_penalty': kl_penalty.item(),
            'train/kl_divergence': kl.mean().item(),
            'train/ratio_mean': ratio.mean().item(),
            'train/ratio_std': ratio.std().item(),
            'train/advantages_mean': advantages.mean().item(),
            'train/clipped_fraction': ((ratio < 1.0 - self.config.clip_ratio) | 
                                      (ratio > 1.0 + self.config.clip_ratio)).float().mean().item()
        }
        
        return total_loss, stats
    
    def train_step(self, batch: Dict) -> Dict:
        """Single GRPO training step"""
        
        prompts = batch['prompt']
        vague_reqs = batch['vague_requirement']
        
        # Step 1: Sample G outputs for each question
        print(f"  Sampling {self.config.group_size} outputs per question...")
        generated_groups, input_ids_groups, attention_mask_groups = self.sample_group(
            prompts,
            self.config.group_size
        )
        
        # Step 2: Compute rewards
        print(f"  Computing ISO compliance rewards...")
        rewards, reward_stats = self.compute_rewards(generated_groups)
        
        # Step 3: Compute advantages
        advantages = self.compute_advantages(rewards)
        
        # Step 4: Compute old policy log probs (before update)
        print(f"  Computing log probabilities...")
        self.policy_model.eval()
        old_log_probs_list = []
        ref_log_probs_list = []
        
        for question_idx in range(len(prompts)):
            for sample_idx in range(self.config.group_size):
                input_ids = input_ids_groups[question_idx][sample_idx].unsqueeze(0)
                attention_mask = attention_mask_groups[question_idx][sample_idx].unsqueeze(0)
                
                # Old policy log probs
                old_log_prob = self.compute_log_probs(
                    self.policy_model,
                    input_ids,
                    attention_mask
                )
                old_log_probs_list.append(old_log_prob)
                
                # Reference policy log probs
                ref_log_prob = self.compute_log_probs(
                    self.ref_model,
                    input_ids,
                    attention_mask
                )
                ref_log_probs_list.append(ref_log_prob)
        
        old_log_probs = torch.cat(old_log_probs_list)
        ref_log_probs = torch.cat(ref_log_probs_list)
        
        # Flatten advantages
        advantages_flat = torch.tensor(
            [adv for group in advantages for adv in group],
            dtype=torch.float32,
            device=self.device
        )
        
        # Step 5: Split into minibatches and train
        print(f"  Training on {self.config.num_minibatches} minibatches...")
        self.policy_model.train()
        
        total_samples = len(prompts) * self.config.group_size
        minibatch_size = total_samples // self.config.num_minibatches
        
        epoch_stats = []
        
        for epoch in range(self.config.inner_epochs):
            # Shuffle indices
            indices = torch.randperm(total_samples)
            
            for mb in range(self.config.num_minibatches):
                start_idx = mb * minibatch_size
                end_idx = start_idx + minibatch_size
                mb_indices = indices[start_idx:end_idx]
                
                # Get minibatch data
                mb_advantages = advantages_flat[mb_indices]
                mb_old_log_probs = old_log_probs[mb_indices]
                mb_ref_log_probs = ref_log_probs[mb_indices]
                
                # Recompute policy log probs for current policy
                mb_policy_log_probs = []
                for idx in mb_indices:
                    question_idx = idx // self.config.group_size
                    sample_idx = idx % self.config.group_size
                    
                    input_ids = input_ids_groups[question_idx][sample_idx].unsqueeze(0)
                    attention_mask = attention_mask_groups[question_idx][sample_idx].unsqueeze(0)
                    
                    # Forward pass through policy
                    outputs = self.policy_model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        return_dict=True
                    )
                    
                    logits = outputs.logits
                    shift_logits = logits[:, :-1, :].contiguous()
                    shift_labels = input_ids[:, 1:].contiguous()
                    
                    log_probs = F.log_softmax(shift_logits, dim=-1)
                    token_log_probs = torch.gather(
                        log_probs,
                        dim=-1,
                        index=shift_labels.unsqueeze(-1)
                    ).squeeze(-1)
                    
                    mask = attention_mask[:, 1:].contiguous()
                    token_log_probs = token_log_probs * mask
                    sequence_log_prob = token_log_probs.sum(dim=-1)
                    
                    mb_policy_log_probs.append(sequence_log_prob)
                
                mb_policy_log_probs = torch.cat(mb_policy_log_probs)
                
                # Compute loss
                loss, loss_stats = self.compute_grpo_loss(
                    mb_policy_log_probs,
                    mb_ref_log_probs,
                    mb_old_log_probs,
                    mb_advantages
                )
                
                # Backward pass
                loss.backward()
                
                # Gradient clipping
                torch.nn.utils.clip_grad_norm_(
                    self.policy_model.parameters(),
                    self.config.max_grad_norm
                )
                
                # Optimizer step
                self.optimizer.step()
                self.optimizer.zero_grad()
                
                epoch_stats.append(loss_stats)
        
        # Aggregate statistics
        aggregated_stats = {
            key: np.mean([stat[key] for stat in epoch_stats])
            for key in epoch_stats[0].keys()
        }
        
        # Add reward statistics
        aggregated_stats.update({
            'reward/mean': reward_stats['mean_reward'],
            'reward/std': reward_stats['std_reward'],
            'reward/pass_rate': reward_stats['pass_rate'],
            'reward/mean_rules_passed': reward_stats['mean_rules_passed']
        })
        
        return aggregated_stats
    
    def evaluate(self, eval_dataset: Dataset) -> Dict:
        """Evaluate model on held-out data"""
        self.policy_model.eval()
        
        eval_loader = DataLoader(
            eval_dataset,
            batch_size=1,
            shuffle=False
        )
        
        total_reward = 0.0
        total_passed = 0
        total_rules_passed = 0
        num_samples = 0
        
        examples = []
        
        with torch.no_grad():
            for batch in tqdm(eval_loader, desc="Evaluating"):
                if num_samples >= self.config.eval_samples:
                    break
                
                prompt = batch['prompt'][0]
                vague_req = batch['vague_requirement'][0]
                
                # Generate single output
                prompt_inputs = self.tokenizer(
                    prompt,
                    return_tensors="pt",
                    padding=True,
                    truncation=True
                ).to(self.device)
                
                outputs = self.policy_model.generate(
                    **prompt_inputs,
                    max_new_tokens=self.config.max_seq_length // 2,
                    temperature=0.7,
                    do_sample=True,
                    top_p=0.95,
                    pad_token_id=self.tokenizer.pad_token_id
                )
                
                full_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
                generated_req = full_text[len(prompt):].strip()
                
                # Compute reward
                reward, rule_results = compute_iso_compliance_reward(generated_req)
                rules_passed = sum(rule_results.values())
                
                total_reward += reward
                if reward >= self.config.reward_threshold:
                    total_passed += 1
                total_rules_passed += rules_passed
                num_samples += 1
                
                # Save examples
                if len(examples) < 5:
                    examples.append({
                        'vague': vague_req,
                        'generated': generated_req,
                        'reward': reward,
                        'rules_passed': f"{rules_passed}/42"
                    })
        
        eval_stats = {
            'eval/mean_reward': total_reward / num_samples,
            'eval/pass_rate': total_passed / num_samples,
            'eval/mean_rules_passed': total_rules_passed / num_samples / 42.0
        }
        
        # Log examples to wandb
        wandb.log({
            "eval/examples": wandb.Table(
                columns=["Vague", "Generated", "Reward", "Rules Passed"],
                data=[[ex['vague'], ex['generated'], ex['reward'], ex['rules_passed']] 
                      for ex in examples]
            )
        })
        
        return eval_stats
    
    def update_reference_model(self):
        """Update reference model with current policy"""
        print("Updating reference model...")
        self.ref_model.load_state_dict(self.policy_model.state_dict())
        self.ref_model.eval()
    
    def save_checkpoint(self, step: int, is_best: bool = False):
        """Save model checkpoint"""
        save_dir = Path(self.config.output_dir) / f"checkpoint-{step}"
        save_dir.mkdir(parents=True, exist_ok=True)
        
        self.policy_model.save_pretrained(save_dir)
        self.tokenizer.save_pretrained(save_dir)
        
        if is_best:
            best_dir = Path(self.config.output_dir) / "best_model"
            best_dir.mkdir(parents=True, exist_ok=True)
            self.policy_model.save_pretrained(best_dir)
            self.tokenizer.save_pretrained(best_dir)
    
    def train(
        self,
        train_dataset: Dataset,
        eval_dataset: Optional[Dataset] = None
    ):
        """Main training loop"""
        
        train_loader = DataLoader(
            train_dataset,
            batch_size=self.config.num_questions_per_step,
            shuffle=True,
            collate_fn=self._collate_fn
        )
        
        print(f"Starting GRPO training for {self.config.max_steps} steps...")
        print(f"Config: LR={self.config.learning_rate}, KL={self.config.kl_coef}, "
              f"Clip={self.config.clip_ratio}, Group Size={self.config.group_size}")
        
        for step in range(self.config.max_steps):
            self.global_step = step
            
            print(f"\n{'='*60}")
            print(f"Step {step + 1}/{self.config.max_steps}")
            print(f"{'='*60}")
            
            # Get batch
            try:
                batch = next(iter(train_loader))
            except StopIteration:
                train_loader = DataLoader(
                    train_dataset,
                    batch_size=self.config.num_questions_per_step,
                    shuffle=True,
                    collate_fn=self._collate_fn
                )
                batch = next(iter(train_loader))
            
            # Training step
            train_stats = self.train_step(batch)
            
            # Log to wandb
            wandb.log({**train_stats, 'step': step})
            
            # Print statistics
            print(f"\nTraining Statistics:")
            print(f"  Loss: {train_stats['loss/total']:.4f}")
            print(f"  Reward: {train_stats['reward/mean']:.4f} "
                  f"(Pass Rate: {train_stats['reward/pass_rate']:.2%})")
            print(f"  Rules Passed: {train_stats['reward/mean_rules_passed']:.1f}/42")
            print(f"  KL Divergence: {train_stats['train/kl_divergence']:.4f}")
            
            # Evaluation
            if eval_dataset and (step + 1) % self.config.eval_interval == 0:
                print(f"\nRunning evaluation...")
                eval_stats = self.evaluate(eval_dataset)
                wandb.log({**eval_stats, 'step': step})
                
                print(f"\nEvaluation Statistics:")
                print(f"  Reward: {eval_stats['eval/mean_reward']:.4f}")
                print(f"  Pass Rate: {eval_stats['eval/pass_rate']:.2%}")
                print(f"  Rules Passed: {eval_stats['eval/mean_rules_passed']:.2%}")
                
                # Save best model
                if eval_stats['eval/mean_reward'] > self.best_eval_score:
                    self.best_eval_score = eval_stats['eval/mean_reward']
                    self.save_checkpoint(step, is_best=True)
                    print(f"  ✓ New best model saved!")
            
            # Update reference model
            if (step + 1) % self.config.ref_model_update_interval == 0:
                self.update_reference_model()
            
            # Save checkpoint
            if (step + 1) % self.config.save_interval == 0:
                self.save_checkpoint(step)
        
        print("\nTraining completed!")
        self.save_checkpoint(self.config.max_steps - 1)
    
    def _collate_fn(self, batch):
        """Collate function for DataLoader"""
        return {
            'prompt': [item['prompt'] for item in batch],
            'vague_requirement': [item['vague_requirement'] for item in batch],
            'requirement_id': [item['requirement_id'] for item in batch]
        }


def main():
    """Main training script"""
    
    # Configuration
    config = GRPOConfig(
        model_name="path/to/your/best-dpo-model",  # Your best DPO checkpoint
        learning_rate=3e-6,
        kl_coef=0.001,
        clip_ratio=10.0,
        group_size=16,
        num_questions_per_step=32,
        max_steps=2000,
        output_dir="./grpo_requirements_output"
    )
    
    # Load datasets
    print("Loading datasets...")
    train_dataset = RequirementsDataset(
        data_path="data/train_requirements.json",
        tokenizer=AutoTokenizer.from_pretrained(config.model_name)
    )
    
    eval_dataset = RequirementsDataset(
        data_path="data/eval_requirements.json",
        tokenizer=AutoTokenizer.from_pretrained(config.model_name)
    )
    
    print(f"Train samples: {len(train_dataset)}")
    print(f"Eval samples: {len(eval_dataset)}")
    
    # Initialize trainer
    trainer = GRPOTrainer(config)
    
    # Train
    trainer.train(train_dataset, eval_dataset)


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'iso_compliance_reward'